In [ ]:
from datasets import load_dataset

import re
from string import punctuation
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

In [2]:
raw = load_dataset("csv", data_files="cw_data.csv", column_names=["text", "label"])
dataset = raw["train"]

In [3]:
class StemCleaner:  
    NORMALISATION_MAP = {
        'acct': 'account',    # ~45 occurrences
        'accts': 'accounts',  # 1 occurrence
        'pls': 'please',      # ~12 occurrences
        'plz': 'please',
        'pwd': 'password',    # 1 occurrence
        'wout': 'without',    # w/out after punctuation removal
        'emial': 'email',     # 8 occurrences
        'wat': 'what',        # ~38 occurrences
        '2': 'to',            # ~30 occurrences
        'cant': 'cannot',     # 7 occurrences — expands so negation is preserved
        'didnt': 'did not',   # 1 occurrence
        'doesnt': 'does not',
        'dont': 'do not',
        'wont': 'will not',
        'isnt': 'is not',
    }
    
    def __init__(self, data, norm=False, num=False, stop=False):
        self.data = data
        self.negations = {'not', 'no', 'cannot'}
        self.stop_words = set(stopwords.words('english')) - self.negations

        self.lowered = self.lowercase(self.data)
        self.punctuated = self.remove_punctuation(self.lowered)
        self.normalised = self.normalise_text(self.punctuated) if norm else self.punctuated
        self.tokenised = self.tokenise(self.normalised)
        self.numerics_removed = self.remove_standalone_numerics(self.tokenised) if num else self.tokenised
        self.cleaned = self.remove_stop_words(self.numerics_removed) if stop else self.numerics_removed
        self.stemmed = self.stem(self.cleaned)
        self.rejoined = self.rejoin(self.stemmed)

    def lowercase(self, texts):
        return [t.lower() for t in texts]
    
    def remove_punctuation(self, texts):
        extra = '—'  
        all_punct = punctuation + extra
        cleaned_texts = []
        for t in texts:
            t = t.translate(str.maketrans('', '', all_punct))
            cleaned_texts.append(t)
        return cleaned_texts
    
    def normalise_text(self, texts):
        normalised = []
        for t in texts:
            for informal, formal in self.NORMALISATION_MAP.items():
                t = re.sub(r'\b' + re.escape(informal) + r'\b', formal, t)
            normalised.append(t)
        return normalised
    
    def tokenise(self, texts):
        return [word_tokenize(t) for t in texts]
    
    def remove_standalone_numerics(self, tokenised_texts):
        cleaned = []
        for t in tokenised_texts:
            cleaned.append([w for w in t if not re.fullmatch(r'\d+', w)])
        return cleaned
    
    def remove_stop_words(self, tokenised_texts):
        cleaned_texts = []
        for t in tokenised_texts:
            cleaned_texts.append([w for w in t if w not in self.stop_words])
        return cleaned_texts
    
    def stem(self, cleaned_texts):
        porter = PorterStemmer()
        stemmed_texts = []
        for t in cleaned_texts:
            stemmed_texts.append([porter.stem(w) for w in t])
        return stemmed_texts
    
    def rejoin(self, stemmed_texts):
        return [' '.join(tokens) for tokens in stemmed_texts]

In [4]:
parsed = StemCleaner(dataset["text"], norm=False, num=False, stop=True)
parsed.stemmed[:5]

[['track', 'show', 'ship', 'no', 'deliveri', 'help', 'pl'],
 ['partial', 'shipment', 'track', 'separ'],
 ['invalid', 'email', 'enter', 'wat', 'happen'],
 ['recov', 'acct', 'suspend'],
 ['emial', 'updat', 'fail', 'error', 'fix']]

In [ ]:
#vectorizer = TfidfVectorizer(ngram_range=(1, 2))
vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
y = dataset["label"]

train_texts, test_texts, y_train, y_test = train_test_split(
    parsed.rejoined, y, test_size=0.2, random_state=42, stratify=y
)

X_train = vectorizer.fit_transform(train_texts)   # fit only on train
X_test = vectorizer.transform(test_texts)  

In [6]:
model = LinearSVC(class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

LinearSVC(class_weight='balanced', random_state=42)

In [7]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

                         precision    recall  f1-score   support

       account_recovery       0.80      1.00      0.89         4
        billing_history       1.00      0.75      0.86         4
           data_privacy       1.00      0.50      0.67         4
         delete_account       0.83      1.00      0.91         5
            login_issue       1.00      1.00      1.00         3
  notification_settings       1.00      1.00      1.00         5
         order_tracking       0.75      0.75      0.75         4
          payment_issue       1.00      0.86      0.92         7
         refund_request       0.75      0.75      0.75         4
         reset_password       1.00      1.00      1.00         4
         shipping_delay       0.86      1.00      0.92         6
subscription_management       0.71      1.00      0.83         5
        two_factor_auth       1.00      0.50      0.67         2
           update_email       1.00      1.00      1.00         5

               accuracy

In [8]:
scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro')
print(f"Macro F1: {scores.mean():.3f} (+/- {scores.std():.3f})")

Macro F1: 0.895 (+/- 0.043)


In [9]:
errors = [(text, true, pred) 
          for text, true, pred in zip(dataset["text"], y_test, y_pred) 
          if true != pred]

for text, true, pred in errors:
    print(f"Text: {text}")
    print(f"True: {true} | Predicted: {pred}\n")

Text: tracking shows shipped but no delivery help pls
True: order_tracking | Predicted: shipping_delay

Text: view personal data and download copy?
True: billing_history | Predicted: subscription_management

Text: billing history not updating… wat do?
True: payment_issue | Predicted: refund_request

Text: cant pay with credit card immediately?
True: refund_request | Predicted: subscription_management

Text: expedite account recovery possible?
True: two_factor_auth | Predicted: account_recovery

Text: I'm unable to sign in — is this because my account is locked?
True: data_privacy | Predicted: order_tracking

Text: delete acct from France possible?
True: data_privacy | Predicted: delete_account

